# 论文 28：用于开放域问答的稠密段落检索
## Vladimir Karpukhin、Barlas Oğuz、Sewon Min 等人，Meta AI (2020)

### 稠密段落检索 (DPR)

学习问题和段落的稠密嵌入。通过嵌入空间的相似性进行检索。性能超过 BM25！

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import re

np.random.seed(42)

## 双编码器架构

```
Question → Encoder_Q → q (dense vector)
Passage  → Encoder_P → p (dense vector)

Similarity: sim(q, p) = q · p  (dot product)
```

In [ ]:
class SimpleTextEncoder:
    '简化的文本编码器（实际应用中：使用 BERT）'
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        
        # 嵌入
        self.embeddings = np.random.randn(vocab_size, embedding_dim) * 0.01
        
        # 简单 RNN 权重
        self.W_xh = np.random.randn(hidden_dim, embedding_dim) * 0.01
        self.W_hh = np.random.randn(hidden_dim, hidden_dim) * 0.01
        self.b_h = np.zeros((hidden_dim, 1))
        
        # 输出投影
        self.W_out = np.random.randn(hidden_dim, hidden_dim) * 0.01
    
    def encode(self, token_ids):
        """将token ID 序列编码为稠密向量
        返回：稠密嵌入（hidden_dim，）"""
        h = np.zeros((self.hidden_dim, 1))
        
        # 处理 token
        for token_id in token_ids:
            # 查找嵌入
            x = self.embeddings[token_id].reshape(-1, 1)
            
            # RNN步
            h = np.tanh(np.dot(self.W_xh, x) + np.dot(self.W_hh, h) + self.b_h)
        
        # 最终表示（类似 CLS）
        output = np.dot(self.W_out, h).flatten()
        
        # L2 余弦相似度标准化
        output = output / (np.linalg.norm(output) + 1e-8)
        
        return output

# 创建编码器
vocab_size = 1000
embedding_dim = 64
hidden_dim = 128

question_encoder = SimpleTextEncoder(vocab_size, embedding_dim, hidden_dim)
passage_encoder = SimpleTextEncoder(vocab_size, embedding_dim, hidden_dim)

# 测试
test_tokens = [10, 25, 37, 42]
q_emb = question_encoder.encode(test_tokens)
p_emb = passage_encoder.encode(test_tokens)

print(f"Question embedding shape: {q_emb.shape}")
print(f"Passage embedding shape: {p_emb.shape}")
print(f"Similarity (dot product): {np.dot(q_emb, p_emb):.4f}")

## 合成问答数据集

In [ ]:
class SimpleTokenizer:
    '简单的简单分词器'
    def __init__(self):
        self.word_to_id = {}
        self.id_to_word = {}
        self.next_id = 0
    
    def tokenize(self, text):
        '将文本转换为token ID'
        words = text.lower().split()
        token_ids = []
        
        for word in words:
            if word not in self.word_to_id:
                self.word_to_id[word] = self.next_id
                self.id_to_word[self.next_id] = word
                self.next_id += 1
            token_ids.append(self.word_to_id[word])
        
        return token_ids

# 创建合成数据集
passages = [
    "The Eiffel Tower is a wrought-iron lattice tower in Paris, France.",
    "The Great Wall of China is a series of fortifications in northern China.",
    "The Statue of Liberty is a colossal neoclassical sculpture in New York.",
    "The Colosseum is an oval amphitheatre in the centre of Rome, Italy.",
    "The Taj Mahal is an ivory-white marble mausoleum in Agra, India.",
    "Mount Everest is Earth's highest mountain above sea level.",
    "The Amazon River is the largest river by discharge volume of water.",
    "The Sahara is a desert on the African continent.",
]

questions = [
    ("What is the Eiffel Tower?", 0),  # （问题，relevant_passage_idx）
    ("Where is the Great Wall located?", 1),
    ("What is the tallest mountain?", 5),
    ("Where is the Statue of Liberty?", 2),
    ("What is the largest river?", 6),
]

# 分词并转换为 token ID
tokenizer = SimpleTokenizer()

passage_tokens = [tokenizer.tokenize(p) for p in passages]
question_tokens = [(tokenizer.tokenize(q), idx) for q, idx in questions]

print("Sample passage:")
print(f"Text: {passages[0]}")
print(f"Tokens: {passage_tokens[0][:10]}...")
print(f"\nVocabulary size: {tokenizer.next_id}")

## 对语料库和问题进行编码

In [ ]:
# 使用正确的词汇大小重新初始化编码器
vocab_size = tokenizer.next_id
question_encoder = SimpleTextEncoder(vocab_size, embedding_dim=32, hidden_dim=64)
passage_encoder = SimpleTextEncoder(vocab_size, embedding_dim=32, hidden_dim=64)

# 对所有段落进行编码
passage_embeddings = []
for tokens in passage_tokens:
    emb = passage_encoder.encode(tokens)
    passage_embeddings.append(emb)
passage_embeddings = np.array(passage_embeddings)

# 对问题进行编码
question_embeddings = []
for tokens, _ in question_tokens:
    emb = question_encoder.encode(tokens)
    question_embeddings.append(emb)
question_embeddings = np.array(question_embeddings)

print(f"Passage embeddings: {passage_embeddings.shape}")
print(f"Question embeddings: {question_embeddings.shape}")

## 通过最大内积搜索进行稠密检索 (MIPS)

In [ ]:
def retrieve_top_k(query_embedding, passage_embeddings, k=3):
    """检索 top-k 段落进行查询
    使用点积相似度 (MIPS)"""
    # 计算相似度
    similarities = np.dot(passage_embeddings, query_embedding)
    
    # 获取 top-k 索引
    top_k_indices = np.argsort(similarities)[::-1][:k]
    top_k_scores = similarities[top_k_indices]
    
    return top_k_indices, top_k_scores

# 测试检索
print("\nDense Retrieval Results:\n" + "="*80)
for i, (q_tokens, correct_idx) in enumerate(question_tokens):
    question_text = questions[i][0]
    q_emb = question_embeddings[i]
    
    # 执行检索
    top_indices, top_scores = retrieve_top_k(q_emb, passage_embeddings, k=3)
    
    print(f"\nQ: {question_text}")
    print(f"Correct passage: #{correct_idx}")
    print(f"\nRetrieved (top-3):")
    for rank, (idx, score) in enumerate(zip(top_indices, top_scores), 1):
        is_correct = "✓" if idx == correct_idx else "✗"
        print(f"  {rank}. [{is_correct}] (score={score:.3f}) {passages[idx][:60]}...")

print("\n" + "="*80)
print("(Encoders are untrained, so results are random)")

## 使用批内负例进行训练

In [ ]:
def softmax(x):
    exp_x = np.exp(x - np.max(x))  # 数值稳定性
    return exp_x / np.sum(exp_x)

def contrastive_loss(query_emb, positive_emb, negative_embs):
    """对比损失 (InfoNCE)
    
    L = -log( exp(q·p+) / (exp(q·p+) + Σ exp(q·p-)) )"""
    # 正例得分
    pos_score = np.dot(query_emb, positive_emb)
    
    # 负例得分
    neg_scores = [np.dot(query_emb, neg_emb) for neg_emb in negative_embs]
    
    # 所有分数
    all_scores = np.array([pos_score] + neg_scores)
    
    # Softmax
    probs = softmax(all_scores)
    
    # 负对数似然（正数应该是第一位）
    loss = -np.log(probs[0] + 1e-8)
    
    return loss

# 模拟训练批次
batch_size = 3
batch_questions = question_embeddings[:batch_size]
batch_passages = passage_embeddings[:batch_size]

# 批内负例：对每个问题，将批次中其他段落作为负例
total_loss = 0
print("\nIn-Batch Negative Training:\n" + "="*80)
for i in range(batch_size):
    q_emb = batch_questions[i]
    pos_emb = batch_passages[i]  # 正确段落
    
    # 负例：批量中的所有其他段落
    neg_embs = [batch_passages[j] for j in range(batch_size) if j != i]
    
    loss = contrastive_loss(q_emb, pos_emb, neg_embs)
    total_loss += loss
    
    print(f"Question {i}: loss = {loss:.4f}")

avg_loss = total_loss / batch_size
print(f"\nAverage batch loss: {avg_loss:.4f}")
print("\nIn-batch negatives: efficient hard negative mining!")

## 可视化嵌入空间

In [ ]:
# 简单的 2D 投影（类似 PCA）
def project_2d(embeddings):
    '将高维嵌入投影到 2D（简化的 PCA）'
    # 平均中心
    mean = np.mean(embeddings, axis=0)
    centered = embeddings - mean
    
    # 取前 2 个主成分（简化）
    U, S, Vt = np.linalg.svd(centered, full_matrices=False)
    projected = U[:, :2] * S[:2]
    
    return projected

# 投影为二维
all_embeddings = np.vstack([passage_embeddings, question_embeddings])
projected = project_2d(all_embeddings)

passage_2d = projected[:len(passage_embeddings)]
question_2d = projected[len(passage_embeddings):]

# 可视化
plt.figure(figsize=(12, 10))

# 绘制段落
plt.scatter(passage_2d[:, 0], passage_2d[:, 1], s=200, c='lightblue', 
           edgecolors='black', linewidths=2, marker='s', label='Passages', zorder=2)

# 注释段落
for i, (x, y) in enumerate(passage_2d):
    plt.text(x, y-0.15, f'P{i}', ha='center', fontsize=10, fontweight='bold')

# 绘制问题
plt.scatter(question_2d[:, 0], question_2d[:, 1], s=200, c='lightcoral', 
           edgecolors='black', linewidths=2, marker='o', label='Questions', zorder=3)

# 注释问题
for i, (x, y) in enumerate(question_2d):
    plt.text(x, y+0.15, f'Q{i}', ha='center', fontsize=10, fontweight='bold')

# 绘制连接（正确段落的问题）
for i, (q_tokens, correct_idx) in enumerate(question_tokens):
    q_pos = question_2d[i]
    p_pos = passage_2d[correct_idx]
    plt.plot([q_pos[0], p_pos[0]], [q_pos[1], p_pos[1]], 
            'g--', alpha=0.5, linewidth=2, label='Correct' if i == 0 else '')

plt.xlabel('Dimension 1', fontsize=12)
plt.ylabel('Dimension 2', fontsize=12)
plt.title('Dense Retrieval Embedding Space (2D Projection)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nIdeal: Questions close to their relevant passages!")

## 与 BM25（稀疏检索）比较

In [ ]:
class SimpleBM25:
    '简化的 BM25 评分'
    def __init__(self, passages, k1=1.5, b=0.75):
        self.passages = passages
        self.k1 = k1
        self.b = b
        
        # 计算文档频率
        self.doc_freqs = {}
        self.avg_doc_len = 0
        
        all_words = []
        for passage in passages:
            words = set(passage.lower().split())
            all_words.extend(passage.lower().split())
            for word in words:
                self.doc_freqs[word] = self.doc_freqs.get(word, 0) + 1
        
        self.avg_doc_len = len(all_words) / len(passages)
        self.N = len(passages)
    
    def score(self, query, passage_idx):
        '计算查询与段落之间的 BM25 得分'
        query_words = query.lower().split()
        passage = self.passages[passage_idx]
        passage_words = passage.lower().split()
        passage_len = len(passage_words)
        
        # 计算术语频率
        tf = Counter(passage_words)
        
        score = 0
        for word in query_words:
            if word not in tf:
                continue
            
            # 逆文档频率（IDF）
            df = self.doc_freqs.get(word, 0)
            idf = np.log((self.N - df + 0.5) / (df + 0.5) + 1)
            
            # TF 分量
            freq = tf[word]
            norm = 1 - self.b + self.b * (passage_len / self.avg_doc_len)
            tf_component = (freq * (self.k1 + 1)) / (freq + self.k1 * norm)
            
            score += idf * tf_component
        
        return score
    
    def retrieve(self, query, k=3):
        '检索 top-k 段落进行查询'
        scores = [self.score(query, i) for i in range(len(self.passages))]
        top_k_indices = np.argsort(scores)[::-1][:k]
        top_k_scores = [scores[i] for i in top_k_indices]
        return top_k_indices, top_k_scores

# 创建 BM25 检索器
bm25 = SimpleBM25(passages)

# 比较 BM25 与 Dense
print("\nBM25 vs Dense Retrieval Comparison:\n" + "="*80)
for i, (question_text, correct_idx) in enumerate(questions):
    print(f"\nQ: {question_text}")
    print(f"Correct: #{correct_idx}")
    
    # BM25
    bm25_indices, bm25_scores = bm25.retrieve(question_text, k=3)
    print(f"\nBM25 Top-3:")
    for rank, (idx, score) in enumerate(zip(bm25_indices, bm25_scores), 1):
        is_correct = "✓" if idx == correct_idx else "✗"
        print(f"  {rank}. [{is_correct}] (score={score:.3f}) #{idx}")
    
    # 稠密
    q_emb = question_embeddings[i]
    dense_indices, dense_scores = retrieve_top_k(q_emb, passage_embeddings, k=3)
    print(f"\nDense Top-3:")
    for rank, (idx, score) in enumerate(zip(dense_indices, dense_scores), 1):
        is_correct = "✓" if idx == correct_idx else "✗"
        print(f"  {rank}. [{is_correct}] (score={score:.3f}) #{idx}")

print("\n" + "="*80)
print("BM25: Lexical matching (sparse)")
print("Dense: Semantic matching (dense embeddings)")

## 检索指标

In [ ]:
def compute_metrics(predictions, correct_indices, k_values=[1, 3, 5]):
    """计算检索指标：
    - Recall@k：正确段落位于 top-k 中的查询百分比
    - MRR（平均倒数排名）：平均 1/正确段落排名"""
    n_queries = len(predictions)
    
    recalls = {k: 0 for k in k_values}
    reciprocal_ranks = []
    
    for pred, correct_idx in zip(predictions, correct_indices):
        # 找到正确段落的排名
        if correct_idx in pred:
            rank = list(pred).index(correct_idx) + 1
            reciprocal_ranks.append(1.0 / rank)
            
            # 更新Recall@k
            for k in k_values:
                if rank <= k:
                    recalls[k] += 1
        else:
            reciprocal_ranks.append(0.0)
    
    # 计算平均值
    mrr = np.mean(reciprocal_ranks)
    recalls = {k: v / n_queries for k, v in recalls.items()}
    
    return recalls, mrr

# 评估两种方法
bm25_predictions = []
dense_predictions = []
correct_indices = []

for i, (question_text, correct_idx) in enumerate(questions):
    # BM25
    bm25_top, _ = bm25.retrieve(question_text, k=5)
    bm25_predictions.append(bm25_top)
    
    # 稠密
    q_emb = question_embeddings[i]
    dense_top, _ = retrieve_top_k(q_emb, passage_embeddings, k=5)
    dense_predictions.append(dense_top)
    
    correct_indices.append(correct_idx)

# 计算指标
bm25_recalls, bm25_mrr = compute_metrics(bm25_predictions, correct_indices)
dense_recalls, dense_mrr = compute_metrics(dense_predictions, correct_indices)

# 展示
print("\nRetrieval Metrics:\n" + "="*60)
print(f"{'Metric':<15} {'BM25':<15} {'Dense':<15}")
print("-" * 60)
for k in [1, 3, 5]:
    print(f"Recall@{k:<10} {bm25_recalls[k]:<15.2%} {dense_recalls[k]:<15.2%}")
print(f"MRR{'':<12} {bm25_mrr:<15.3f} {dense_mrr:<15.3f}")
print("="*60)
print("\n(Models are untrained - results are random)")

## 要点总结总结总结总结总结

### 稠密段落检索 (DPR) 架构：

**双编码器**：
```
Question: q → BERT_Q → E_Q(q) = q_emb
Passage:  p → BERT_P → E_P(p) = p_emb

Similarity: sim(q, p) = q_emb · p_emb
```

### 训练目标：

**对比损失 (InfoNCE)**：
$$
L(q_i, p_i^+, p_i^{-1}, ..., p_i^{-n}) = -\log \frac{e^{\text{sim}(q_i, p_i^+)}}{e^{\text{sim}(q_i, p_i^+)} + \sum_j e^{\text{sim}(q_i, p_i^{-j})}}
$$

其中：
- $p_i^+$：正例（相关）段落
- $p_i^{-j}$：负例（不相关）段落

### 批内负例：

高效的负例挖掘：
```
Batch: [(q1, p1+), (q2, p2+), ..., (qB, pB+)]

For q1:
  Positive: p1+
  Negatives: p2+, p3+, ..., pB+ (from other examples)
```

**好处**：
- 不需要额外准备负例段落
- 梯度可以通过批次中的所有样本传播
- 能够扩展到较大的批次

### 困难负例挖掘：

1. **BM25 负例**：排名靠前但不相关的 BM25 结果
2. **随机负例**：来自语料库的随机段落
3. **批内负例**：同一批次中其他样本的正例

**最佳**：将三者结合起来！

### 推理（检索）：

**离线**：
1. 对所有段落进行编码：$P = \{E_P(p_1), ..., E_P(p_N)\}$
2. 构建 MIPS 索引（例如 FAISS）

**在线**（查询时）：
1. 编码查询：$q_{emb} = E_Q(q)$
2. 搜索索引：top-k by $\arg\max_p \, q_{emb} \cdot p_{emb}$

### DPR 与 BM25：

| 方面 | BM25 | DPR |
|--------|------|-----|
| 匹配方式 | 词法匹配（精确词语） | 语义匹配（含义） |
| 训练 | 无（启发式方法） | 从数据中学习 |
| 鲁棒性 | 对措辞敏感 | 能处理释义和改写 |
| 速度 | 快（稀疏向量） | 借助 MIPS 索引实现快速检索 |
| 内存 | 低 | 高（稠密向量） |

### 结果（来自论文）：

**Natural Questions**：
- BM25：Top-20 准确率 59.1%
- DPR：Top-20 准确率 78.4%

**WebQuestions**：
- BM25：55.0%
- DPR：75.0%

**TREC**：
- BM25：70.9%
- DPR：79.4%

### 实现细节：

1. **编码器**：BERT-base（110M 参数）
2. **嵌入维度**：768（BERT 隐藏层维度）
3. **批量大小**：128（较大的批次有利于获得更多批内负例）
4. **困难负例**：每个正例搭配 1 个 BM25 负例和 1 个随机负例
5. **训练**：在 59k 个问答对上训练约 40 个 epoch

### 优点：

- ✅ **语义匹配**：理解含义，而不仅仅是单词
- ✅ **端到端**：从问题-段落对中学习
- ✅ **处理释义和改写**：“最高的山”=“最高峰”
- ✅ **可扩展**：MIPS 带有 FAISS，可用于数十亿个段落
- ✅ **优于 BM25**：+15-20% 绝对准确度

### 限制：

- ❌ **需要训练数据**：需要 QA 对
- ❌ **内存开销**：所有段落的稠密向量
- ❌ **索引更新**：语料库更改时重新编码
- ❌ **可能会错过精确匹配**：BM25 更适合稀有实体

### 最佳实践：

1. **混合检索**：结合 BM25 + DPR
2. **大批次**：更多批内负例
3. **困难负例**：使用 BM25 排名靠前的结果
4. **微调**：特定领域的数据可改善结果
5. **FAISS**：用于大规模快速 MIPS

### 现代扩展：

- **ColBERT**：后期交互以获得更好的排名
- **ANCE**：近似最近邻负例
- **RocketQA**：跨批次负例
- **Contriever**：无监督稠密检索
- **Dense X Retrieval**：多向量表示

### 应用：

- 开放域 QA（例如 Google 搜索）
- RAG（检索增强生成）
- 文献检索
- 语义搜索
- 知识库补全